In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("PySparkExercises").getOrCreate()

In [2]:
from google.colab import files
uploaded=files.upload

In [3]:
# PART 1
stores_df = spark.read.option("header","true").option("inferSchema","true").csv("stores.csv")

In [4]:
products_df = spark.read.option("header","true").option("inferSchema","true").csv("products.csv")

In [5]:
inventory_df = spark.read.option("header","true").option("inferSchema","true").csv("inventory.csv")

In [6]:
sales_df = spark.read.option("header","true").option("inferSchema","true").csv("sales.csv")

In [7]:
suppliers_df = spark.read.option("multiline","true").json("suppliers.json")

In [8]:
stores_df.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)



In [9]:
print(stores_df.count(),products_df.count(),inventory_df.count(),sales_df.count(),suppliers_df.count())

8 12 12 15 1


In [10]:
stores_df.write.mode("overwrite").parquet("bronze/stores")

In [11]:
# PART 2
from pyspark.sql.functions import *
products_df.filter(col("supplier_id").isNull()).show()

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+



In [12]:
inventory_df.filter(col("stock_quantity").isNull()).show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+



In [13]:
sales_df.filter(col("sale_amount").isNull()).show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL|      UPI|
+-------+--------+----------+----------+-------------+-----------+---------+



In [15]:
sales_df.filter(col("payment_m").isNull()).show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1010|    S101|      P104|2026-01-16|            2|      14000|     NULL|
+-------+--------+----------+----------+-------------+-----------+---------+



In [16]:
inventory_df=inventory_df.fillna({"stock_quantity":0})

In [17]:
sales_df=sales_df.fillna({"sale_amount":0})

In [18]:
sales_df=sales_df.fillna({"payment_m":"Not Provided"})

In [19]:
products_df=products_df.fillna({"supplier_id":"UNKNOWN"})

In [20]:
products_df=products_df.withColumn("data_quality_status",when(col("supplier_id")=="UNKNOWN","Incomplete").otherwise("Complete"))

In [21]:
products_df.write.mode("overwrite").parquet("silver/products")

In [25]:
# PART 3
suppliers_df = spark.read.option("multiline", "true").schema(supplier_schema).json("suppliers.json")
suppliers_flat = suppliers_df.select("supplier_id", "supplier_name", "city", "rating", col("contact.phone").alias("phone"), col("contact.email").alias("email"))

In [26]:
suppliers_flat.select("phone").show()

+-----+
|phone|
+-----+
| NULL|
+-----+



In [27]:
suppliers_flat.select("email").show()

+-----+
|email|
+-----+
| NULL|
+-----+



In [28]:
suppliers_flat=suppliers_flat.fillna({"phone":"Not Provided"})

In [29]:
suppliers_flat=suppliers_flat.fillna({"email":"Not Provided"})

In [30]:
suppliers_flat.write.mode("overwrite").parquet("silver/suppliers")

In [31]:
# PART 4
products_suppliers=products_df.join(suppliers_flat,"supplier_id","left")

In [32]:
inventory_products=inventory_df.join(products_df,"product_id","left")

In [33]:
sales_stores=sales_df.join(stores_df,"store_id","left")

In [34]:
sales_products=sales_df.join(products_df,"product_id","left")

In [35]:
retail_df=sales_df.join(products_df,"product_id","left").join(stores_df,"store_id","left").join(suppliers_flat,"supplier_id","left")

In [36]:
products_df.join(suppliers_flat,"supplier_id","left_anti").show()

+-----------+----------+------------+-----------+------------+----------+-------------------+
|supplier_id|product_id|product_name|   category|       brand|unit_price|data_quality_status|
+-----------+----------+------------+-----------+------------+----------+-------------------+
|       S201|      P101|      Laptop|Electronics|      Lenovo|     65000|           Complete|
|       S202|      P102|      Mobile|Electronics|     Samsung|     25000|           Complete|
|       S203|      P103|  Television|Electronics|          LG|     45000|           Complete|
|       S204|      P104|Office Chair|  Furniture| Featherlite|      7000|           Complete|
|       S204|      P105| Study Table|  Furniture|Urban Ladder|     12000|           Complete|
|       S205|      P106|       Shoes|    Fashion|        Nike|      4500|           Complete|
|       S206|      P107|       Watch|    Fashion|    Fastrack|      8000|           Complete|
|       S206|      P108|    Backpack|    Fashion|   Wildcraf

In [37]:
inventory_df.join(products_df,"product_id","left_anti").show()

+----------+------------+--------+--------------+-------------+-----------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|
+----------+------------+--------+--------------+-------------+-----------+
|      P120|       I1012|    S108|            12|            5| 2026-01-18|
+----------+------------+--------+--------------+-------------+-----------+



In [38]:
sales_df.join(products_df,"product_id","left_anti").show()

+----------+-------+--------+----------+-------------+-----------+---------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_m|
+----------+-------+--------+----------+-------------+-----------+---------+
|      P120| SA1009|    S108|2026-01-15|            2|      10000|     Cash|
+----------+-------+--------+----------+-------------+-----------+---------+



In [39]:
sales_df.join(stores_df,"store_id","left_anti").show()

+--------+-------+----------+---------+-------------+-----------+---------+
|store_id|sale_id|product_id|sale_date|quantity_sold|sale_amount|payment_m|
+--------+-------+----------+---------+-------------+-----------+---------+
+--------+-------+----------+---------+-------------+-----------+---------+



In [59]:
# PART 5
retail_df=sales_df.join(products_df,"product_id","left")\
                    .join(stores_df,"store_id","left")\
                    .join(suppliers_flat.withColumnRenamed("city", "supplier_city"),"supplier_id","left")\
                    .join(inventory_df,["product_id","store_id"],"left")
retail_df=retail_df.withColumn("stock_status",when(col("stock_quantity")<=col("reorder_level"),"Reorder Required").otherwise("Sufficient Stock"))

In [42]:
retail_df=retail_df.withColumn("price_category",when(col("unit_price")>=50000,"Premium").when(col("unit_price")>=10000,"Standard").otherwise("Budget"))

In [43]:
retail_df=retail_df.withColumn("revenue_category",when(col("sale_amount")>=50000,"High Revenue").when(col("sale_amount")>=15000,"Medium Revenue").otherwise("Low Revenue"))

In [44]:
retail_df=retail_df.withColumn("month",month(col("sale_date")))

In [45]:
retail_df=retail_df.withColumn("year",year(col("sale_date")))

In [46]:
retail_df=retail_df.withColumn("inventory_value",col("stock_quantity")*col("unit_price"))

In [47]:
retail_df=retail_df.withColumn("supplier_quality",when(col("rating")>=4.5,"Excellent").when(col("rating")>=4.0,"Good").otherwise("Average"))

In [48]:
# PART 6
stores_df.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    1|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    1|
|  Rajasthan|    1|
|  Telangana|    1|
|Maharashtra|    2|
+-----------+-----+



In [49]:
products_df.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|    Fashion|    4|
|Electronics|    5|
|  Furniture|    3|
+-----------+-----+



In [50]:
products_df.groupBy("brand").count().show()

+------------+-----+
|       brand|count|
+------------+-----+
|        Nike|    1|
|        Sony|    1|
|Urban Ladder|    1|
|        Puma|    1|
|      Lenovo|    1|
| Featherlite|    1|
|     Samsung|    1|
|      Godrej|    1|
|          LG|    1|
|   Wildcraft|    1|
|    Fastrack|    1|
|   Whirlpool|    1|
+------------+-----+



In [51]:
retail_df.groupBy("store_id").agg(sum("inventory_value")).show()

+--------+--------------------+
|store_id|sum(inventory_value)|
+--------+--------------------+
|    S105|              250000|
|    S102|              745000|
|    S106|                   0|
|    S104|               64000|
|    S107|               32000|
|    S101|             1921000|
|    S108|                NULL|
|    S103|              159000|
+--------+--------------------+



In [52]:
retail_df.groupBy("category").agg(sum("inventory_value")).show()

+-----------+--------------------+
|   category|sum(inventory_value)|
+-----------+--------------------+
|    Fashion|              449000|
|       NULL|                NULL|
|Electronics|             2645000|
|  Furniture|               77000|
+-----------+--------------------+



In [53]:
retail_df.filter(col("stock_status")=="Reorder Required").count()

6

In [54]:
retail_df.agg(sum("sale_amount").alias("total_revenue")).show()

+-------------+
|total_revenue|
+-------------+
|       373000|
+-------------+



In [55]:
retail_df.groupBy("store_name").agg(sum("sale_amount")).show()

+--------------------+----------------+
|          store_name|sum(sale_amount)|
+--------------------+----------------+
|Metro Mart Bangalore|           65000|
|    Metro Mart Kochi|           32000|
|   Metro Mart Jaipur|           10000|
|   Metro Mart Mumbai|           30000|
|     Metro Mart Pune|           38000|
|Metro Mart Hyderabad|          154000|
|    Metro Mart Delhi|           20000|
|  Metro Mart Chennai|           24000|
+--------------------+----------------+



In [60]:
retail_df.groupBy(retail_df.city).agg(sum("sale_amount")).show()

+---------+----------------+
|     city|sum(sale_amount)|
+---------+----------------+
|Bangalore|           65000|
|    Kochi|           32000|
|  Chennai|           24000|
|   Mumbai|           30000|
|     Pune|           38000|
|    Delhi|           20000|
|Hyderabad|          154000|
|   Jaipur|           10000|
+---------+----------------+



In [61]:
retail_df.groupBy("category").agg(sum("sale_amount")).show()

+-----------+----------------+
|   category|sum(sale_amount)|
+-----------+----------------+
|    Fashion|           62000|
|       NULL|           10000|
|Electronics|          243000|
|  Furniture|           58000|
+-----------+----------------+



In [62]:
retail_df.groupBy("product_name").agg(sum("sale_amount")).show()

+------------+----------------+
|product_name|sum(sale_amount)|
+------------+----------------+
|Office Chair|           14000|
|        NULL|           10000|
|Refrigerator|           38000|
|      Laptop|          130000|
|        Sofa|           32000|
|    Backpack|           20000|
|       Shoes|           18000|
|      Mobile|           75000|
|  Television|               0|
| Study Table|           12000|
|       Watch|           24000|
+------------+----------------+



In [64]:
retail_df.groupBy("payment_m").agg(sum("sale_amount")).show()

+------------+----------------+
|   payment_m|sum(sale_amount)|
+------------+----------------+
|        Card|          133000|
|        Cash|           35500|
|Not Provided|           14000|
|         UPI|          190500|
+------------+----------------+



In [65]:
retail_df.groupBy("product_name").agg(sum("sale_amount").alias("revenue")).orderBy(desc("revenue")).show(1)

+------------+-------+
|product_name|revenue|
+------------+-------+
|      Laptop| 130000|
+------------+-------+
only showing top 1 row


In [66]:
retail_df.groupBy("store_name").agg(sum("sale_amount").alias("revenue")).orderBy(desc("revenue")).show(1)

+--------------------+-------+
|          store_name|revenue|
+--------------------+-------+
|Metro Mart Hyderabad| 154000|
+--------------------+-------+
only showing top 1 row


In [67]:
retail_df.groupBy("category").agg(sum("sale_amount").alias("revenue")).orderBy(desc("revenue")).show(1)

+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 243000|
+-----------+-------+
only showing top 1 row


In [70]:
# PART 7
from pyspark.sql.window import Window
w=Window.orderBy(desc("sale_amount"))

In [71]:
retail_df.withColumn("rank",rank().over(w)).show()

+----------+--------+-----------+-------+----------+-------------+-----------+------------+------------+-----------+------------+----------+-------------------+--------------------+---------+-----------+-----------+------------+-------------+-------------+------+-----+-----+------------+--------------+-------------+-----------+----------------+----+
|product_id|store_id|supplier_id|sale_id| sale_date|quantity_sold|sale_amount|   payment_m|product_name|   category|       brand|unit_price|data_quality_status|          store_name|     city|      state| store_type|manager_name|supplier_name|supplier_city|rating|phone|email|inventory_id|stock_quantity|reorder_level|last_update|    stock_status|rank|
+----------+--------+-----------+-------+----------+-------------+-----------+------------+------------+-----------+------------+----------+-------------------+--------------------+---------+-----------+-----------+------------+-------------+-------------+------+-----+-----+------------+--------

In [72]:
store_rev=retail_df.groupBy("store_name").agg(sum("sale_amount").alias("revenue"))

In [73]:
store_rev.withColumn("rank",rank().over(Window.orderBy(desc("revenue")))).show()

+--------------------+-------+----+
|          store_name|revenue|rank|
+--------------------+-------+----+
|Metro Mart Hyderabad| 154000|   1|
|Metro Mart Bangalore|  65000|   2|
|     Metro Mart Pune|  38000|   3|
|    Metro Mart Kochi|  32000|   4|
|   Metro Mart Mumbai|  30000|   5|
|  Metro Mart Chennai|  24000|   6|
|    Metro Mart Delhi|  20000|   7|
|   Metro Mart Jaipur|  10000|   8|
+--------------------+-------+----+



In [74]:
category_product_rev=retail_df.groupBy("category","product_name").agg(sum("sale_amount").alias("revenue"))

In [75]:
category_product_rev.withColumn("rank",rank().over(Window.partitionBy("category").orderBy(desc("revenue")))).show()

+-----------+------------+-------+----+
|   category|product_name|revenue|rank|
+-----------+------------+-------+----+
|       NULL|        NULL|  10000|   1|
|Electronics|      Laptop| 130000|   1|
|Electronics|      Mobile|  75000|   2|
|Electronics|Refrigerator|  38000|   3|
|Electronics|  Television|      0|   4|
|    Fashion|       Watch|  24000|   1|
|    Fashion|    Backpack|  20000|   2|
|    Fashion|       Shoes|  18000|   3|
|  Furniture|        Sofa|  32000|   1|
|  Furniture|Office Chair|  14000|   2|
|  Furniture| Study Table|  12000|   3|
+-----------+------------+-------+----+



In [76]:
category_product_rev.withColumn("rank",rank().over(Window.partitionBy("category").orderBy(desc("revenue")))).filter(col("rank")==1).show()

+-----------+------------+-------+----+
|   category|product_name|revenue|rank|
+-----------+------------+-------+----+
|       NULL|        NULL|  10000|   1|
|Electronics|      Laptop| 130000|   1|
|    Fashion|       Watch|  24000|   1|
|  Furniture|        Sofa|  32000|   1|
+-----------+------------+-------+----+



In [77]:
category_product_rev.withColumn("rank",rank().over(Window.partitionBy("category").orderBy(desc("revenue")))).filter(col("rank")<=3).show()

+-----------+------------+-------+----+
|   category|product_name|revenue|rank|
+-----------+------------+-------+----+
|       NULL|        NULL|  10000|   1|
|Electronics|      Laptop| 130000|   1|
|Electronics|      Mobile|  75000|   2|
|Electronics|Refrigerator|  38000|   3|
|    Fashion|       Watch|  24000|   1|
|    Fashion|    Backpack|  20000|   2|
|    Fashion|       Shoes|  18000|   3|
|  Furniture|        Sofa|  32000|   1|
|  Furniture|Office Chair|  14000|   2|
|  Furniture| Study Table|  12000|   3|
+-----------+------------+-------+----+



In [78]:
state_store_rev=retail_df.groupBy("state","store_name").agg(sum("sale_amount").alias("revenue"))

In [79]:
state_store_rev.withColumn("rank",rank().over(Window.partitionBy("state").orderBy(desc("revenue")))).filter(col("rank")==1).show()

+-----------+--------------------+-------+----+
|      state|          store_name|revenue|rank|
+-----------+--------------------+-------+----+
|      Delhi|    Metro Mart Delhi|  20000|   1|
|  Karnataka|Metro Mart Bangalore|  65000|   1|
|     Kerala|    Metro Mart Kochi|  32000|   1|
|Maharashtra|     Metro Mart Pune|  38000|   1|
|  Rajasthan|   Metro Mart Jaipur|  10000|   1|
| Tamil Nadu|  Metro Mart Chennai|  24000|   1|
|  Telangana|Metro Mart Hyderabad| 154000|   1|
+-----------+--------------------+-------+----+



In [80]:
daily_rev=retail_df.groupBy("sale_date").agg(sum("sale_amount").alias("daily_revenue"))

In [81]:
daily_rev.withColumn("running_total",sum("daily_revenue").over(Window.orderBy("sale_date"))).show()

+----------+-------------+-------------+
| sale_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2026-01-10|       115000|       115000|
|2026-01-11|        65000|       180000|
|2026-01-12|        26000|       206000|
|2026-01-13|        12500|       218500|
|2026-01-14|        38000|       256500|
|2026-01-15|        42000|       298500|
|2026-01-16|        14000|       312500|
|2026-01-17|            0|       312500|
|2026-01-18|        12000|       324500|
|2026-02-01|        16000|       340500|
|2026-02-02|         7500|       348000|
|2026-02-03|        25000|       373000|
+----------+-------------+-------------+



In [82]:
daily_rev.withColumn("previous_day_sales",lag("daily_revenue").over(Window.orderBy("sale_date"))).show()

+----------+-------------+------------------+
| sale_date|daily_revenue|previous_day_sales|
+----------+-------------+------------------+
|2026-01-10|       115000|              NULL|
|2026-01-11|        65000|            115000|
|2026-01-12|        26000|             65000|
|2026-01-13|        12500|             26000|
|2026-01-14|        38000|             12500|
|2026-01-15|        42000|             38000|
|2026-01-16|        14000|             42000|
|2026-01-17|            0|             14000|
|2026-01-18|        12000|                 0|
|2026-02-01|        16000|             12000|
|2026-02-02|         7500|             16000|
|2026-02-03|        25000|              7500|
+----------+-------------+------------------+



In [83]:
retail_df.withColumn("next_sale_amount",lead("sale_amount").over(Window.orderBy("sale_date"))).show()

+----------+--------+-----------+-------+----------+-------------+-----------+------------+------------+-----------+------------+----------+-------------------+--------------------+---------+-----------+-----------+------------+-------------+-------------+------+-----+-----+------------+--------------+-------------+-----------+----------------+----------------+
|product_id|store_id|supplier_id|sale_id| sale_date|quantity_sold|sale_amount|   payment_m|product_name|   category|       brand|unit_price|data_quality_status|          store_name|     city|      state| store_type|manager_name|supplier_name|supplier_city|rating|phone|email|inventory_id|stock_quantity|reorder_level|last_update|    stock_status|next_sale_amount|
+----------+--------+-----------+-------+----------+-------------+-----------+------------+------------+-----------+------------+----------+-------------------+--------------------+---------+-----------+-----------+------------+-------------+-------------+------+-----+---

In [84]:
retail_df.withColumn("previous_sale",lag("sale_amount").over(Window.orderBy("sale_date"))).filter(col("sale_amount")>col("previous_sale")).show()

+----------+--------+-----------+-------+----------+-------------+-----------+------------+------------+-----------+------------+----------+-------------------+--------------------+---------+-----------+-----------+------------+-------------+-------------+------+-----+-----+------------+--------------+-------------+-----------+----------------+-------------+
|product_id|store_id|supplier_id|sale_id| sale_date|quantity_sold|sale_amount|   payment_m|product_name|   category|       brand|unit_price|data_quality_status|          store_name|     city|      state| store_type|manager_name|supplier_name|supplier_city|rating|phone|email|inventory_id|stock_quantity|reorder_level|last_update|    stock_status|previous_sale|
+----------+--------+-----------+-------+----------+-------------+-----------+------------+------------+-----------+------------+----------+-------------------+--------------------+---------+-----------+-----------+------------+-------------+-------------+------+-----+-----+---

In [85]:
# PART 8
stores_df.createOrReplaceTempView("stores")

In [86]:
products_df.createOrReplaceTempView("products")

In [87]:
inventory_df.createOrReplaceTempView("inventory")

In [88]:
sales_df.createOrReplaceTempView("sales")

In [89]:
suppliers_flat.createOrReplaceTempView("suppliers")

In [90]:
spark.sql("select * from sales").show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|         UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|
| SA1010|    S101|      P104|2026-01-16|

In [91]:
spark.sql("select category,count(*) from products group by category").show()

+-----------+--------+
|   category|count(1)|
+-----------+--------+
|    Fashion|       4|
|Electronics|       5|
|  Furniture|       3|
+-----------+--------+



In [92]:
spark.sql("select store_id,sum(sale_amount) from sales group by store_id").show()

+--------+----------------+
|store_id|sum(sale_amount)|
+--------+----------------+
|    S105|           20000|
|    S102|           65000|
|    S106|           38000|
|    S104|           24000|
|    S107|           32000|
|    S101|          154000|
|    S108|           10000|
|    S103|           30000|
+--------+----------------+



In [93]:
spark.sql("select s.city,sum(sa.sale_amount) from sales sa join stores s on sa.store_id=s.store_id group by s.city").show()

+---------+----------------+
|     city|sum(sale_amount)|
+---------+----------------+
|Bangalore|           65000|
|    Kochi|           32000|
|  Chennai|           24000|
|   Mumbai|           30000|
|     Pune|           38000|
|    Delhi|           20000|
|Hyderabad|          154000|
|   Jaipur|           10000|
+---------+----------------+



In [94]:
spark.sql("select * from inventory where stock_quantity<=reorder_level").show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1003|    S101|      P104|             3|            5| 2026-01-11|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|
|       I1010|    S106|      P109|             0|            6| 2026-01-16|
|       I1011|    S107|      P110|             1|            3| 2026-01-17|
+------------+--------+----------+--------------+-------------+-----------+



In [95]:
spark.sql("select * from sales s left anti join products p on s.product_id=p.product_id").show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1009|    S108|      P120|2026-01-15|            2|      10000|     Cash|
+-------+--------+----------+----------+-------------+-----------+---------+



In [96]:
spark.sql("select * from products p left anti join suppliers s on p.supplier_id=s.supplier_id").show()

+----------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|           Complete|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|           Complete|
|      P103|  Television|Electronics|          LG|       S203|     45000|           Complete|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|           Complete|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|           Complete|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|           Complete|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|           Complete|
|      P108|    Backpack|    Fashion|   Wildcraft|       S20

In [97]:
spark.sql("select product_id,sum(sale_amount) revenue from sales group by product_id order by revenue desc limit 5").show()

+----------+-------+
|product_id|revenue|
+----------+-------+
|      P101| 130000|
|      P102|  75000|
|      P109|  38000|
|      P110|  32000|
|      P107|  24000|
+----------+-------+



In [101]:
spark.sql("select payment_m,sum(sale_amount) from sales group by payment_m").show()

+------------+----------------+
|   payment_m|sum(sale_amount)|
+------------+----------------+
|        Card|          133000|
|        Cash|           35500|
|Not Provided|           14000|
|         UPI|          190500|
+------------+----------------+



In [102]:
# PART 9
retail_df.write.mode("overwrite").parquet("gold/retail_sales")

In [104]:
retail_df = retail_df.withColumn("year", year(col("sale_date"))).withColumn("month", month(col("sale_date")))
retail_df.write.mode("overwrite").partitionBy("year","month").parquet("gold/retail_sales_partitioned")

In [107]:
incremental_sales=spark.read.option("header","true").csv("sales_2026_03.csv")

In [108]:
incremental_sales.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1016|    S101|      P101|2026-03-01|            1|      65000|         UPI|
| SA1017|    S102|      P102|2026-03-02|            2|      50000|        Card|
| SA1018|    S103|      P106|2026-03-03|            3|      13500|        Cash|
| SA1019|    S104|      P107|2026-03-04|            1|       8000|         UPI|
| SA1020|    S105|      P108|2026-03-05|            2|       5000|        Card|
+-------+--------+----------+----------+-------------+-----------+------------+



In [109]:
incremental_sales.write.mode("append").parquet("silver/sales")

In [110]:
silver_sales=spark.read.parquet("silver/sales")

In [111]:
silver_sales.groupBy("product_id").agg(sum("sale_amount").alias("total_revenue")).show()

+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|      P102|      50000.0|
|      P106|      13500.0|
|      P107|       8000.0|
|      P101|      65000.0|
|      P108|       5000.0|
+----------+-------------+



In [112]:
silver_sales.groupBy("store_id").agg(sum("sale_amount").alias("total_revenue")).show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S105|       5000.0|
|    S102|      50000.0|
|    S104|       8000.0|
|    S101|      65000.0|
|    S103|      13500.0|
+--------+-------------+



In [113]:
silver_sales.groupBy("store_id").agg(sum("sale_amount").alias("total_revenue")).show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S105|       5000.0|
|    S102|      50000.0|
|    S104|       8000.0|
|    S101|      65000.0|
|    S103|      13500.0|
+--------+-------------+



In [114]:
retail_df.write.mode("overwrite").partitionBy("year","month").parquet("gold/retail_sales_partitioned")

In [115]:
print(sales_df.count(), silver_sales.count())

15 5


In [125]:
# PART 10
store_performance = retail_df.groupBy("store_id","store_name","city","state").agg(count("*"),sum("sale_amount"))
store_performance.show()

+--------+--------------------+---------+-----------+--------+----------------+
|store_id|          store_name|     city|      state|count(1)|sum(sale_amount)|
+--------+--------------------+---------+-----------+--------+----------------+
|    S106|     Metro Mart Pune|     Pune|Maharashtra|       1|           38000|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala|       1|           32000|
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|       4|          154000|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|       2|           30000|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|       2|           20000|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|       2|           65000|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|       2|           24000|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|       1|           10000|
+--------+--------------------+---------+-----------+--------+----------------+



In [126]:
product_performance = retail_df.groupBy("product_id","product_name","category","brand").agg(sum("quantity_sold"),sum("sale_amount"))
product_performance.show()

+----------+------------+-----------+------------+------------------+----------------+
|product_id|product_name|   category|       brand|sum(quantity_sold)|sum(sale_amount)|
+----------+------------+-----------+------------+------------------+----------------+
|      P110|        Sofa|  Furniture|      Godrej|                 1|           32000|
|      P104|Office Chair|  Furniture| Featherlite|                 2|           14000|
|      P101|      Laptop|Electronics|      Lenovo|                 2|          130000|
|      P109|Refrigerator|Electronics|   Whirlpool|                 1|           38000|
|      P105| Study Table|  Furniture|Urban Ladder|                 1|           12000|
|      P107|       Watch|    Fashion|    Fastrack|                 3|           24000|
|      P102|      Mobile|Electronics|     Samsung|                 3|           75000|
|      P108|    Backpack|    Fashion|   Wildcraft|                 8|           20000|
|      P106|       Shoes|    Fashion|      

In [127]:
inventory_reorder = retail_df.select("store_id","product_id","product_name","stock_quantity","reorder_level","stock_status")
inventory_reorder.show()

+--------+----------+------------+--------------+-------------+----------------+
|store_id|product_id|product_name|stock_quantity|reorder_level|    stock_status|
+--------+----------+------------+--------------+-------------+----------------+
|    S101|      P101|      Laptop|            10|            5|Sufficient Stock|
|    S101|      P102|      Mobile|            25|           10|Sufficient Stock|
|    S102|      P101|      Laptop|             8|            5|Sufficient Stock|
|    S103|      P106|       Shoes|            30|           10|Sufficient Stock|
|    S104|      P107|       Watch|             4|            5|Reorder Required|
|    S105|      P108|    Backpack|            50|           20|Sufficient Stock|
|    S106|      P109|Refrigerator|             0|            6|Reorder Required|
|    S107|      P110|        Sofa|             1|            3|Reorder Required|
|    S108|      P120|        NULL|            12|            5|Sufficient Stock|
|    S101|      P104|Office 

In [128]:
retail_df=retail_df.withColumn("supplier_quality",when(col("rating")>=4.5,"Excellent").when(col("rating")>=4.0,"Good").otherwise("Average"))
supplier_quality_report = retail_df.select("supplier_id","supplier_name","city","rating","supplier_quality","phone","email")
supplier_quality_report.show()

+-----------+-------------+---------+------+----------------+-----+-----+
|supplier_id|supplier_name|     city|rating|supplier_quality|phone|email|
+-----------+-------------+---------+------+----------------+-----+-----+
|       S201|         NULL|Hyderabad|  NULL|         Average| NULL| NULL|
|       S202|         NULL|Hyderabad|  NULL|         Average| NULL| NULL|
|       S201|         NULL|Bangalore|  NULL|         Average| NULL| NULL|
|       S205|         NULL|   Mumbai|  NULL|         Average| NULL| NULL|
|       S206|         NULL|  Chennai|  NULL|         Average| NULL| NULL|
|       S206|         NULL|    Delhi|  NULL|         Average| NULL| NULL|
|       S203|         NULL|     Pune|  NULL|         Average| NULL| NULL|
|       S204|         NULL|    Kochi|  NULL|         Average| NULL| NULL|
|       NULL|         NULL|   Jaipur|  NULL|         Average| NULL| NULL|
|       S204|         NULL|Hyderabad|  NULL|         Average| NULL| NULL|
|       S203|         NULL|Bangalore| 

In [129]:
from pyspark.sql.functions import countDistinct, sum, col
category_revenue = retail_df.groupBy("category").agg(countDistinct("product_id"),sum("quantity_sold"),sum("sale_amount"))
category_revenue.show()

+-----------+--------------------------+------------------+----------------+
|   category|count(DISTINCT product_id)|sum(quantity_sold)|sum(sale_amount)|
+-----------+--------------------------+------------------+----------------+
|    Fashion|                         3|                15|           62000|
|       NULL|                         1|                 2|           10000|
|Electronics|                         4|                 7|          243000|
|  Furniture|                         3|                 4|           58000|
+-----------+--------------------------+------------------+----------------+



In [130]:
payment_mode_report = retail_df.groupBy("payment_m").agg(count("*"),sum("sale_amount"))
payment_mode_report.show()

+------------+--------+----------------+
|   payment_m|count(1)|sum(sale_amount)|
+------------+--------+----------------+
|        Card|       5|          133000|
|        Cash|       3|           35500|
|Not Provided|       1|           14000|
|         UPI|       6|          190500|
+------------+--------+----------------+

